In [ ]:
!pip install gliner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.7 MB/s eta 0:00:00


In [ ]:
# Core
import torch
import pandas as pd
import numpy as np

# NER
from gliner import GLiNER

# Utils
from typing import List, Dict


In [ ]:
# Example: loading tweets from Excel
df = pd.read_excel("/content/trump_tweets_sota_classified(1).xlsx")

# Adjust column name if needed
texts = df["tweet_text"].dropna().tolist()

print(f"Loaded {len(texts)} tweets")


Loaded 255 tweets


In [ ]:
# Load pre-trained GLiNER model
model = GLiNER.from_pretrained("urchade/gliner_medium")

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"GLiNER loaded on {device}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


GLiNER loaded on cpu


In [ ]:
NER_LABELS = [
    "PERSON",
    "ORG",
    "COUNTRY",
    "LOCATION",
    "COMMODITY",
    "CURRENCY",
    "SECTOR",
    "EVENT"
]


In [ ]:
def extract_entities(
    text: str,
    labels: List[str] = NER_LABELS,
    threshold: float = 0.35
) -> List[Dict]:
    """
    Run NER on a single tweet_text.
    Returns list of entity dictionaries.
    """
    if not isinstance(text, str) or not text.strip():
        return []

    preds = model.predict_entities(
        text,
        labels=labels,
        threshold=threshold
    )

    return [
        {
            "entity": p["text"],
            "label": p["label"],
            "score": round(p["score"], 3),
            "start": p["start"],
            "end": p["end"]
        }
        for p in preds
    ]


In [ ]:
sample = df.loc[0, "tweet_text"]
sample


'washingtonexaminer.com/restori'

In [ ]:
extract_entities(sample)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[{'entity': 'washingtonexaminer.com/restori',
  'label': 'ORG',
  'score': 0.534,
  'start': 0,
  'end': 30}]

In [ ]:
df["entities"] = df["tweet_text"].apply(extract_entities)


In [ ]:
df[["tweet_id", "tweet_text", "entities"]].head(5)


,tweet_id,tweet_text,entities
0,f6b6bc55c8b5,washingtonexaminer.com/restori,"[{'entity': 'washingtonexaminer.com/restori', ..."
1,362a7010b3c1,breitbart.com/politics/2025/06,[]
2,65c19d0b7fb5,redstate.com/redstate-guest-ed,[]
3,8f0d572a979f,foxnews.com/opinion/loeffler-t,"[{'entity': 'foxnews.com', 'label': 'ORG', 'sc..."
4,707c1b8d12dd,nypost.com/2025/06/28/us-news/,"[{'entity': 'nypost.com', 'label': 'ORG', 'sco..."


In [ ]:
flat_entities = []

for _, row in df.iterrows():
    for ent in row["entities"]:
        flat_entities.append({
            "tweet_id": row["tweet_id"],
            "entity": ent["entity"].lower(),
            "label": ent["label"],
            "score": ent["score"]
        })

ner_df = pd.DataFrame(flat_entities)
ner_df.head(10)
ner_df


,tweet_id,entity,label,score
0,f6b6bc55c8b5,washingtonexaminer.com/restori,ORG,0.534
1,8f0d572a979f,foxnews.com,ORG,0.479
2,8f0d572a979f,loeffler-t,PERSON,0.513
3,707c1b8d12dd,nypost.com,ORG,0.875
4,9e27945fb8cb,faith leaders conference call,EVENT,0.980
...,...,...,...,...
826,68c3cb276bab,idaho,LOCATION,0.990
827,68c3cb276bab,state,LOCATION,0.776
828,68c3cb276bab,2016,EVENT,0.565
829,68c3cb276bab,2020,EVENT,0.636


In [ ]:
ner_df.groupby(["label", "entity"]) \
      .size() \
      .sort_values(ascending=False) \
      .head(30)


label     entity         
COUNTRY   iran               44
          america            18
ORG       truthsocial.com    13
COUNTRY   israel             12
          united states      11
LOCATION  los angeles        10
ORG       youtube             8
COUNTRY   china               8
          u.s.                8
          country             7
PERSON    trump               6
LOCATION  nuclear sites       6
EVENT     peace               6
ORG       national guard      5
          military            5
EVENT     deal                5
LOCATION  country             5
PERSON    president trump     5
          biden               4
          donald trump        4
ORG       fbi                 4
          republicans         4
          nato                4
          maga                4
CURRENCY  bill                4
ORG       army                4
          cnn                 4
          congress            4
PERSON    djt                 4
          donald j. trump     4
dtype: int64